# Structured Feature Setup: ETO PARAT + Compustat front end

Shared front end for the two structured dimensions (technology, people). It
resolves the listed firms of the active universe (`UNIVERSE`, Fortune 500 or
S&P 500) to ETO PARAT company IDs and to Compustat FY2024 gvkeys, assembles
one wide raw-input table with the four computed shares and per-dimension
completeness flags, and builds the HTML input review. The cached outputs
(`universe_listed`, `eto_matched`, `wrds_matched`, `inputs`) are consumed by
the per-dimension notebooks, which write the indicator parquets (one row per
firm, NaN wherever an input is missing — marked, not dropped).

Run this notebook once per universe before `technology.ipynb` or
`people.ipynb` (same `UNIVERSE` value in all three).

**Sources.**
- `data_raw/eto/core.csv` — PARAT company aggregates: AI vs. total
  publications and patents (lifetime totals), AI / Tech Team 1 workforce
  (~2024 snapshot). `ticker.csv` and `alias.csv` support the match.
- `data_raw/wrds/fundamentals_annual.csv` — Compustat annual fundamentals;
  only FY2024 `emp` (employees, in thousands) is needed here.

**Prerequisites.** For the fortune500 universe the CIK matching pass reuses
the EDGAR filing resolution cached by `nlp_features_setup.ipynb` (falls back
to ticker/name if missing). The sp500 universe carries gvkeys natively, so
its Compustat match is a direct join.

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import sys
import webbrowser
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.universe import load_universe
from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.structured_features import (
    INDICATOR,
    PEOPLE_FEATURES,
    TARGET_FYEAR,
    TECHNOLOGY_FEATURES,
    build_inputs,
    build_structured_inputs_review,
    match_to_eto,
    match_to_wrds,
    normalize_ticker,
)

# Firm universe this run builds ("fortune500" or "sp500"). Every cache,
# indicator parquet, and validation artifact is scoped by this value, so
# both universes coexist side by side.
UNIVERSE = "sp500"

# Cached steps under data_cache/indicators/<UNIVERSE>/structured_features/.
# All sources are local CSVs, so a full recompute is cheap; the caches mainly
# pin the matched universe for the dimension notebooks:
#   universe_listed - listed firms of the universe            (section 1)
#   eto_matched     - universe -> ETO PARAT company IDs       (section 2)
#   wrds_matched    - universe -> Compustat FY2024 gvkeys     (section 3)
#   inputs          - raw inputs + shares + flags per firm    (section 4)
# Toggle FORCE_REFRESH to recompute all, or clear_cache(INDICATOR, UNIVERSE, "<step>").
FORCE_REFRESH = False
SHARED = INDICATOR

VALIDATION_DIR = PROJECT_ROOT / "data_clean" / "validation" / UNIVERSE / SHARED
print(f"Universe: {UNIVERSE}  |  cache namespace: {SHARED}  |  Compustat fiscal year: {TARGET_FYEAR}")

Universe: sp500  |  cache namespace: structured_features  |  Compustat fiscal year: 2024


## 1. Universe: listed firms

Fortune 500: top 500 of the Fortune 1000 US list, privately held firms
(ticker "~") excluded. S&P 500: constituents are listed by construction, so
nothing drops. Both dimensions target this set; firms missing an input keep
their rows with NaN indicators — marked, never dropped.

In [2]:
universe_df = None if FORCE_REFRESH else load_cached_step(SHARED, "universe_listed", UNIVERSE)
if universe_df is None:
    universe_df = load_universe(UNIVERSE)
    universe_df = universe_df[universe_df["ticker"].map(normalize_ticker) != ""].reset_index(drop=True)
    save_cached_step(universe_df, SHARED, "universe_listed", UNIVERSE)
    print(f"Resolved {len(universe_df)} listed firms (saved to {cache_path(SHARED, 'universe_listed', UNIVERSE)})")
else:
    print(f"Loaded {len(universe_df)} listed firms from cache ({cache_path(SHARED, 'universe_listed', UNIVERSE)})")

assert universe_df["ticker"].is_unique, "duplicate tickers in the universe"
assert universe_df["normalized_company_name"].is_unique, "duplicate normalized names in the universe"
universe_df.head()

Loaded 500 listed firms from cache (D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\sp500\structured_features\universe_listed.parquet)


,rank,company,ticker,normalized_company_name,gvkey,cik
0,1,3M Company,MMM,3m,007435,0000066740
1,2,A. O. Smith Corporation,AOS,a o smith,009771,0000091142
2,3,"AMETEK, Inc.",AME,ametek,001598,0001037868
3,4,APA Corporation,APA,apa,001678,0001841666
4,5,AT&T Inc.,T,att,009899,0000732717


## 2. Match to ETO PARAT

Three ordered passes, first unambiguous hit wins: ticker (`ticker.csv`),
normalized company name (`core.csv`), normalized alias (`alias.csv`). A key
mapping to more than one PARAT ID within a pass is never picked silently —
the firm falls through to the next pass and ends up `ambiguous` if nothing
else resolves it. If several firms land on the same PARAT ID, only the
strongest match method keeps it (ticker > name > alias): PARAT aliases
include former company names, so weaker duplicates are usually historical
collisions (e.g. SAIC hitting Leidos via its pre-spin-off name) rather than
real subsidiary aggregation.

Unmatched firms are either not covered by PARAT (it tracks ~700 companies
with significant AI activity) or corporate-action successors (FedEx
Freight, Honeywell Aerospace, Paramount Skydance, ...) whose PARAT
predecessor describes a different corporate scope — both stay unmatched by
design and carry NaN in both dimensions.

In [3]:
eto_matched = None if FORCE_REFRESH else load_cached_step(SHARED, "eto_matched", UNIVERSE)
if eto_matched is None:
    eto_matched = match_to_eto(universe_df)
    save_cached_step(eto_matched, SHARED, "eto_matched", UNIVERSE)
    print(f"Matched {int(eto_matched['eto_id'].notna().sum())} firms (saved to {cache_path(SHARED, 'eto_matched', UNIVERSE)})")
else:
    print(f"Loaded ETO match from cache ({cache_path(SHARED, 'eto_matched', UNIVERSE)})")

print("\nMatch method counts:")
print(eto_matched["eto_match_method"].value_counts().to_string())

_named = eto_matched.merge(
    universe_df[["normalized_company_name", "ticker", "company"]], on="normalized_company_name"
)
print("\nName/alias matches (verify these pairs by eye):")
print(_named.loc[_named["eto_match_method"].isin(["name", "alias"]),
                 ["ticker", "company", "eto_name"]].to_string(index=False))

_amb = _named[_named["eto_match_method"] == "ambiguous"]
print(f"\n{len(_amb)} ambiguous (candidate collision, never silently picked):")
print(_amb[["ticker", "company"]].to_string(index=False) if len(_amb) else "  none")

_un = _named[_named["eto_match_method"] == "unmatched"]
print(f"\n{len(_un)} firms without a PARAT record:")
print(", ".join(_un["ticker"].tolist()))

Loaded ETO match from cache (D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\sp500\structured_features\eto_matched.parquet)

Match method counts:
eto_match_method
ticker       439
unmatched     42
name          13
alias          5
manual         1

Name/alias matches (verify these pairs by eye):
ticker                                 company                           eto_name
   APA                         APA Corporation                    APA Corporation
  ABNB                            Airbnb, Inc.                             Airbnb
  BALL                        Ball Corporation                               Ball
   COR                           Cencora, Inc.                            Cencora
     C                          Citigroup Inc.                               Citi
   XOM                 Exxon Mobil Corporation                         ExxonMobil
   BEN                Franklin Resources, Inc.                 Fran

## 3. Match to Compustat (WRDS, FY2024)

`fundamentals_annual.csv` is filtered to FY2024 and collapsed to one row per
gvkey (industrial format preferred over financial-services twins, then USD,
standard consolidated data, populated `emp`, largest assets). The sp500
universe carries gvkeys natively, so the match is a direct join; fortune500
runs the CIK > ticker > normalized-name cascade with the same
never-silently-pick and duplicate-demotion rules as the ETO match. `emp` is
reported in thousands; `employees_wrds` is the converted headcount and the
denominator of both People shares.

In [4]:
wrds_matched = None if FORCE_REFRESH else load_cached_step(SHARED, "wrds_matched", UNIVERSE)
if wrds_matched is None:
    wrds_matched = match_to_wrds(universe_df, universe=UNIVERSE)
    save_cached_step(wrds_matched, SHARED, "wrds_matched", UNIVERSE)
    print(f"Matched {int(wrds_matched['gvkey'].notna().sum())} firms (saved to {cache_path(SHARED, 'wrds_matched', UNIVERSE)})")
else:
    print(f"Loaded WRDS match from cache ({cache_path(SHARED, 'wrds_matched', UNIVERSE)})")

print("\nMatch method counts:")
print(wrds_matched["wrds_match_method"].value_counts().to_string())

_wnamed = wrds_matched.merge(
    universe_df[["normalized_company_name", "ticker", "company"]], on="normalized_company_name"
)
_wun = _wnamed[_wnamed["wrds_match_method"].isin(["unmatched", "ambiguous"])]
print(f"\n{len(_wun)} firms without a usable Compustat FY{TARGET_FYEAR} row:")
print(_wun[["ticker", "company", "wrds_match_method"]].to_string(index=False) if len(_wun) else "  none")

_noemp = _wnamed[_wnamed["gvkey"].notna()
                 & (_wnamed["employees_wrds"].isna() | _wnamed["employees_wrds"].eq(0))]
print(f"\n{len(_noemp)} matched firms with missing/zero employees (People shares NaN):")
print(_noemp[["ticker", "company", "wrds_conm"]].to_string(index=False) if len(_noemp) else "  none")

Loaded WRDS match from cache (D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\sp500\structured_features\wrds_matched.parquet)

Match method counts:
wrds_match_method
gvkey    500

0 firms without a usable Compustat FY2024 row:
  none

2 matched firms with missing/zero employees (People shares NaN):
ticker                             company                   wrds_conm
  FDXF FedEx Freight Holding Company, Inc. FEDEX FRGHT HLDNG CMPNY INC
  HONA            Honeywell Aerospace Inc.     HONEYWELL AEROSPACE INC


## 4. Combined input table + sanity checks

One wide row per firm: ids, match provenance, all raw inputs, the four
computed shares (NaN wherever an input is missing or a denominator is
zero — a firm with zero total patents has no defined AI patent share, which
is not the same as a genuine 0), and the `technology_complete` /
`people_complete` flags that mark which firms have every input. The
cross-check recomputes each share against
ETO's own rounded percentage columns, which validates both the join (right
company row) and the arithmetic in one shot.

In [5]:
inputs = None if FORCE_REFRESH else load_cached_step(SHARED, "inputs", UNIVERSE)
if inputs is None:
    inputs = build_inputs(universe_df, eto_matched, wrds_matched)
    save_cached_step(inputs, SHARED, "inputs", UNIVERSE)
    print(f"Built input table for {len(inputs)} firms (saved to {cache_path(SHARED, 'inputs', UNIVERSE)})")
else:
    print(f"Loaded input table from cache ({cache_path(SHARED, 'inputs', UNIVERSE)})")

print(f"\ntechnology_complete: {int(inputs['technology_complete'].sum())}/{len(inputs)}")
print(f"people_complete:     {int(inputs['people_complete'].sum())}/{len(inputs)}")

# Cross-check against ETO's own percentage columns (rounded to one decimal).
_m = inputs[inputs["eto_id"].notna()]
_dev_pub = (_m["ai_publication_share"] * 100 - _m["eto_ai_publication_pct"]).abs()
_dev_pat = (_m["ai_patent_share"] * 100 - _m["eto_ai_patent_pct"]).abs()
print(f"\ncross-check vs ETO percentages: max deviation "
      f"pub {_dev_pub.max():.3f}pp / patent {_dev_pat.max():.3f}pp")
assert _dev_pub.max() < 0.15 and _dev_pat.max() < 0.15, "share arithmetic disagrees with ETO"

# x1000 scaling spot check against well-known headcounts.
_spot = inputs[inputs["ticker"].isin(["WMT", "AMZN", "UNH"])]
print("\nemployees_wrds spot check (Walmart ~2.1M):")
print(_spot[["ticker", "company_name", "employees_wrds"]].to_string(index=False))

_gt1 = inputs[(inputs[PEOPLE_FEATURES] > 1).any(axis=1)]
print(f"\nworker share > 1: {len(_gt1)} firm(s)")
if len(_gt1):
    print(_gt1[["ticker", "company_name"] + PEOPLE_FEATURES].to_string(index=False))
_gt05 = inputs[(inputs[PEOPLE_FEATURES] > 0.5).any(axis=1)]
print(f"worker share > 0.5: {len(_gt05)} firm(s)")
if len(_gt05):
    print(_gt05[["ticker", "company_name"] + PEOPLE_FEATURES].to_string(index=False))

Loaded input table from cache (D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\sp500\structured_features\inputs.parquet)

technology_complete: 284/500
people_complete:     458/500

cross-check vs ETO percentages: max deviation pub 0.050pp / patent 0.050pp

employees_wrds spot check (Walmart ~2.1M):
ticker                    company_name  employees_wrds
  AMZN                Amazon.com, Inc.       1556000.0
   UNH UnitedHealth Group Incorporated        400000.0
   WMT                    Walmart Inc.       2100000.0

worker share > 1: 0 firm(s)
worker share > 0.5: 0 firm(s)


## 5. Manual input review (HTML)

Build one self-contained HTML page with a row per listed firm of the
universe: match provenance for both sources, every raw input, the four
computed shares, and the complete / missing status per dimension. **Missing
inputs and zero denominators are flagged red** (the affected indicators are
NaN — marked, not dropped); worker shares above 1 are flagged orange. The
page opens automatically in your browser and reads only the cached frame,
so it is safe to re-run any time.

In [6]:
# Self-load from cache so this runs standalone after a kernel restart.
try:
    inputs
except NameError:
    inputs = load_cached_step(SHARED, "inputs", UNIVERSE)
    if inputs is None:
        raise RuntimeError("Run section 4 first — the inputs cache is missing.")

review_path = VALIDATION_DIR / "inputs_review.html"
build_structured_inputs_review(inputs, review_path)
print(f"Wrote {review_path}")
print(f"{int((~inputs['technology_complete']).sum())} firms flagged incomplete for Technology, "
      f"{int((~inputs['people_complete']).sum())} for People.")
webbrowser.open(review_path.resolve().as_uri())

Wrote D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_clean\validation\sp500\structured_features\inputs_review.html
216 firms flagged incomplete for Technology, 42 for People.


True